# YOLO26n Pose 손 21개 관절 학습 (Colab)

Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하고 위에서부터 실행하세요.
기본 데이터는 **MediaPipe로 자동 라벨링한 Ultralytics Hand Keypoints**입니다. Google이 MediaPipe 모델 학습에 사용한 원본 데이터셋이라는 뜻은 아닙니다.
`yolo26n-pose.pt`를 손 관절 21개와 `hand` 1클래스로 파인튜닝합니다. 데이터 ZIP은 자동 다운로드하므로 직접 업로드하지 않아도 됩니다.

이 공개 데이터에는 **왼손·오른손 클래스가 없습니다**. 따라서 기본 실행에서는 handedness를 학습하지 않고 추론 JSON의 `handedness`는 `null`입니다.
직접 좌우 라벨을 준비한 경우에만 `DATA_SOURCE="custom_handedness"`를 선택해 기존 2클래스 학습을 사용할 수 있습니다.

- 출력: 정규화된 2D 좌표, 관절별 confidence, 손 클래스와 검출 confidence. 3D/world 좌표는 출력하지 않습니다.
- MediaPipe 자동 라벨의 오차를 학습할 수 있으며, 검증 점수는 실제 사람이 표기한 정답에 대한 정확도와 다를 수 있습니다.
- 관절 confidence는 좌표 정확도 보증이나 실제 가려짐 확률이 아닙니다. 기본 손실은 visibility 1과 2를 모두 유효 관절로 취급합니다.
- 공개 데이터의 라이선스는 제공처 기준 CC BY-NC-SA 4.0입니다. 아래 출처에서 이용 조건을 확인하세요.

참고: [데이터 출처](https://docs.ultralytics.com/datasets/pose/hand-keypoints/), [데이터 YAML](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/hand-keypoints.yaml), [Pose](https://docs.ultralytics.com/tasks/pose/), [결과 API](https://docs.ultralytics.com/reference/engine/results/).


## 1. 설치 및 GPU 확인
설치 버전은 학습 결과와 함께 기록합니다. 재현 실행에서는 기록된 버전으로 설치하세요.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"])

from pathlib import Path
import json
import math
import hashlib
import shutil
import zipfile
from collections import Counter
from datetime import datetime, timezone

import torch
import ultralytics
import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from google.colab import files, drive

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임을 GPU로 변경한 후 다시 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. 데이터 선택 및 학습 설정

기본 `DATA_SOURCE="mediapipe_hand_keypoints"`는 MediaPipe로 라벨링된 공개 ZIP(제공처 안내 약 369 MB)을 자동 다운로드합니다.
제공된 train/val 분할을 사용하며 클래스는 `0=hand`입니다. `DATA_ZIP`은 이 모드에서 사용하지 않습니다.

선택적으로 `DATA_SOURCE="custom_handedness"`를 지정하면 직접 만든 ZIP을 업로드하거나 Drive의 `DATA_ZIP` 경로를 사용합니다.
이 경우 클래스는 `0=left_hand, 1=right_hand`이고 사람의 실제 왼손/오른손 기준으로 라벨링해야 합니다.

두 모드 모두 아래 구조를 사용합니다.
```
hand_dataset/
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```
라벨 한 행은 `class cx cy w h + 21 × (x y v)`의 숫자 68개입니다.
좌표는 0~1 정규화, `v=0: 미표기, 1: 가려졌지만 표기됨, 2: 보임`입니다. 배경은 빈 TXT를 사용합니다.
관절 순서는 손목(0), 엄지(1–4), 검지(5–8), 중지(9–12), 약지(13–16), 소지(17–20)입니다.

`SAVE_TO_DRIVE=True`이면 체크포인트를 Drive에 저장합니다. 직접 만든 데이터는 세션/사람 단위로 train/val을 나누세요.


In [ ]:
DATA_SOURCE = "mediapipe_hand_keypoints"  # 또는 "custom_handedness"
DATA_ZIP = ""  # 예: "/content/drive/MyDrive/datasets/hand_dataset.zip"
SAVE_TO_DRIVE = True
EPOCHS = 100
IMGSZ = 640
BATCH = 16  # GPU 메모리 부족 시 8 또는 4로 낮추세요.
WORKERS = 2
SEED = 42

if SAVE_TO_DRIVE or DATA_ZIP.startswith("/content/drive/"):
    drive.mount("/content/drive")

run_name = "yolo26n_hand_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
PROJECT = Path("/content/drive/MyDrive/hand_pose_runs" if SAVE_TO_DRIVE else "/content/hand_pose_runs")
PROJECT.mkdir(parents=True, exist_ok=True)
if DATA_SOURCE not in {"mediapipe_hand_keypoints", "custom_handedness"}:
    raise ValueError("지원하지 않는 DATA_SOURCE입니다.")
NAMES = ({0: "hand"} if DATA_SOURCE == "mediapipe_hand_keypoints"
         else {0: "left_hand", 1: "right_hand"})
KEYPOINT_NAMES = [
    "wrist", "thumb_cmc", "thumb_mcp", "thumb_ip", "thumb_tip",
    "index_mcp", "index_pip", "index_dip", "index_tip",
    "middle_mcp", "middle_pip", "middle_dip", "middle_tip",
    "ring_mcp", "ring_pip", "ring_dip", "ring_tip",
    "pinky_mcp", "pinky_pip", "pinky_dip", "pinky_tip",
]
EDGES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
         (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),
         (15,16),(13,17),(0,17),(17,18),(18,19),(19,20)]

## 3. 데이터 자동 다운로드 / 사용자 ZIP 업로드
기본 모드는 공개 배포 ZIP을 다운로드하고, 사용자 데이터 모드만 업로드 창을 엽니다.
이미지는 Colab 로컬 디스크에서 읽고 결과는 선택에 따라 Drive에 저장합니다.
실행마다 새 폴더에 압축을 풀어 기존 데이터와 섞이지 않게 합니다.

In [ ]:
PUBLIC_DATA_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/hand-keypoints.zip"
if DATA_SOURCE == "mediapipe_hand_keypoints":
    from urllib.request import urlopen
    cache_dir = Path("/content/datasets")
    cache_dir.mkdir(parents=True, exist_ok=True)
    public_zip = cache_dir / "hand-keypoints.zip"
    if not public_zip.is_file() or not zipfile.is_zipfile(public_zip):
        partial = public_zip.with_suffix(".part")
        print("공개 MediaPipe 라벨 데이터 다운로드:", PUBLIC_DATA_URL)
        try:
            with urlopen(PUBLIC_DATA_URL, timeout=120) as response, partial.open("wb") as output:
                shutil.copyfileobj(response, output)
            if not zipfile.is_zipfile(partial):
                raise ValueError("다운로드한 파일이 ZIP이 아닙니다.")
            partial.replace(public_zip)
        finally:
            partial.unlink(missing_ok=True)
    DATA_ZIP = str(public_zip)
elif not DATA_ZIP:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("데이터 ZIP 한 개만 업로드하세요.")
    DATA_ZIP = next(iter(uploaded))
    del uploaded

zip_path = Path(DATA_ZIP)
if not zip_path.is_file() or not zipfile.is_zipfile(zip_path):
    raise ValueError(f"유효한 ZIP 파일이 아닙니다: {zip_path}")
extract_dir = Path("/content/datasets") / run_name
extract_dir.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        target = (extract_dir / member.filename).resolve()
        if not target.is_relative_to(extract_dir.resolve()):
            raise ValueError(f"ZIP 외부 경로: {member.filename}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("ZIP의 심볼릭 링크는 지원하지 않습니다.")
    archive.extractall(extract_dir)

roots = sorted({p.parent for p in extract_dir.rglob("images")
                if p.is_dir() and (p / "train").is_dir() and (p / "val").is_dir()
                and (p.parent / "labels" / "train").is_dir()
                and (p.parent / "labels" / "val").is_dir()})
if len(roots) != 1:
    raise ValueError(f"images/train,val 및 labels/train,val을 가진 데이터 루트가 하나여야 합니다: {roots}")
DATA_ROOT = roots[0]
print("데이터:", DATA_ROOT)

## 4. 라벨·이미지 검사 및 YAML 생성
누락 라벨, 잘못된 좌표, 손 클래스 누락, train/val의 동일 이미지가 있으면 학습 전에 중단합니다.
숫자 검사만으로 해부학적 좌우 또는 관절 순서가 올바른지 확인할 수는 없으므로 다음 미리보기도 확인하세요.

In [ ]:
def parse_pose_label(path):
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        values = [float(v) for v in line.split()]
        context = f"{path}:{line_number}"
        if len(values) != 68 or not all(math.isfinite(v) for v in values):
            raise ValueError(f"{context}: 유한한 숫자 68개가 필요합니다.")
        if values[0] not in NAMES:
            raise ValueError(f"{context}: class는 {list(NAMES)} 중 하나여야 합니다.")
        cx, cy, w, h = values[1:5]
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"{context}: bbox 범위 오류")
        points = [values[i:i+3] for i in range(5, 68, 3)]
        for x, y, v in points:
            if v not in (0, 1, 2) or not (0 <= x <= 1 and 0 <= y <= 1):
                raise ValueError(f"{context}: 관절 좌표 또는 visibility 오류")
        if not any(p[2] > 0 for p in points):
            raise ValueError(f"{context}: 유효한 관절이 없습니다.")
        rows.append((int(values[0]), values[1:5], points))
    return rows

image_paths = {}
annotations = {}
split_hashes = {}
summary = {}
for split in ("train", "val"):
    images_dir = DATA_ROOT / "images" / split
    labels_dir = DATA_ROOT / "labels" / split
    paths = sorted(p for p in images_dir.rglob("*")
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    if not paths:
        raise ValueError(f"{split}: 이미지가 없습니다.")
    counts, visibility, hashes = Counter(), Counter(), set()
    expected_labels = set()
    for path in paths:
        label = labels_dir / path.relative_to(images_dir).with_suffix(".txt")
        if label in expected_labels:
            raise ValueError(f"동일 라벨을 공유하는 이미지 파일명: {path}")
        expected_labels.add(label)
        if not label.is_file():
            raise ValueError(f"라벨 누락 (배경도 빈 TXT 필요): {label}")
        with Image.open(path) as im:
            im.verify()
        hashes.add(hashlib.sha256(path.read_bytes()).hexdigest())
        rows = parse_pose_label(label)
        annotations[path] = rows
        for cls, bbox, points in rows:
            counts[cls] += 1
            visibility.update(int(p[2]) for p in points)
    extra = set(labels_dir.rglob("*.txt")) - expected_labels
    if extra:
        raise ValueError(f"이미지 없는 라벨: {next(iter(extra))}")
    if any(counts[c] == 0 for c in NAMES):
        raise ValueError(f"{split}: 설정된 모든 클래스가 필요합니다. 클래스별 개수: {dict(counts)}")
    image_paths[split] = paths
    split_hashes[split] = hashes
    summary[split] = {"images": len(paths), "hands": dict(counts), "visibility": dict(visibility)}
    print(split, summary[split])
if split_hashes["train"] & split_hashes["val"]:
    raise ValueError("train과 val에 동일한 이미지 파일이 있습니다. 세션 단위로 다시 분리하세요.")

DATA_YAML = DATA_ROOT / "hand_pose.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_ROOT), "train": "images/train", "val": "images/val",
    "names": NAMES, "kpt_shape": [21, 3],
    "kpt_names": {c: KEYPOINT_NAMES for c in NAMES},
    "flip_idx": list(range(21)),
}, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text())

## 5. 정답 라벨 미리보기
손의 좌우와 관절 연결이 맞는지 확인하세요. 초록 점은 보임(v=2), 주황 점은 가려졌지만 표기됨(v=1)입니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
rng = np.random.default_rng(SEED)
preview_paths = image_paths["train"]
indices = rng.choice(len(preview_paths), size=min(6, len(preview_paths)), replace=False)
for ax in axes.flat:
    ax.axis("off")
for ax, index in zip(axes.flat, indices):
    path = preview_paths[index]
    with Image.open(path) as im:
        image = np.asarray(im.convert("RGB"))
    height, width = image.shape[:2]
    ax.imshow(image)
    for cls, bbox, points in annotations[path]:
        p = np.asarray(points)
        xy = p[:, :2] * [width, height]
        for a, b in EDGES:
            if p[a, 2] > 0 and p[b, 2] > 0:
                ax.plot(xy[[a,b], 0], xy[[a,b], 1], color="lime", linewidth=1)
        visible = p[:, 2] > 0
        ax.scatter(xy[visible, 0], xy[visible, 1], s=12,
                   c=["lime" if v == 2 else "orange" for v in p[visible, 2]])
        ax.text(bbox[0]*width, bbox[1]*height, NAMES[cls], color="white",
                bbox=dict(facecolor="black", alpha=0.6))
    ax.set_title(path.name)
plt.tight_layout()
plt.show()

## 6. 학습
좌우 반전은 handedness 라벨 교환이 필요하므로 끕니다. `flip_idx`만으로 클래스가 교환되지는 않습니다.
OOM 발생 시 BATCH를 낮추고 다시 학습하세요. 결과 경로는 실제 trainer의 save_dir에서 가져옵니다.

In [ ]:
model = YOLO("yolo26n-pose.pt")
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, workers=WORKERS, seed=SEED, deterministic=True,
    project=str(PROJECT), name=run_name, exist_ok=False,
    patience=25, save=True, save_period=10, plots=True,
    fliplr=0.0, flipud=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0,
    degrees=15.0, translate=0.1, scale=0.3,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / "weights" / "best.pt"
if not BEST.is_file():
    raise FileNotFoundError(f"학습 완료 체크포인트가 없습니다: {BEST}")
shutil.copy2(DATA_YAML, RUN_DIR / "hand_pose.yaml")
(RUN_DIR / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8")
(RUN_DIR / "dataset_summary.json").write_text(
    json.dumps({"classes": NAMES, "keypoints": KEYPOINT_NAMES, "splits": summary,
                "dataset_zip": str(zip_path), "data_source": DATA_SOURCE,
                "source_url": PUBLIC_DATA_URL if DATA_SOURCE == "mediapipe_hand_keypoints" else None}, ensure_ascii=False, indent=2), encoding="utf-8")
print("최적 가중치:", BEST)

## 7. 검증
검증 mAP는 관절 conf가 정확도 확률로 보정되어 있음을 의미하지 않습니다.
학습에 사용하지 않은 실제 촬영 세션에서도 좌표 오차와 confidence 관계를 확인하세요.

In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ,
                         batch=BATCH, device=0, workers=WORKERS, plots=True,
                         project=str(RUN_DIR), name="validation")
print(metrics.results_dict)
(RUN_DIR / "validation_metrics.json").write_text(
    json.dumps({k: float(v) for k, v in metrics.results_dict.items()}, indent=2),
    encoding="utf-8")
for filename in ("results.png", "confusion_matrix.png"):
    path = RUN_DIR / filename
    if path.exists():
        display(Image.open(path))

## 8. 추론: 손 클래스와 관절별 confidence
검증 이미지 한 장을 기본으로 사용합니다. `SAMPLE_IMAGE`를 바꾸면 다른 이미지도 확인할 수 있습니다.
`BOX_CONF`는 손 검출 임계값, `KEYPOINT_CONF`는 표시할 관절 임계값으로 서로 다릅니다.
아래 임계값은 시작값이며 실측 성능으로 조정해야 합니다. JSON에는 필터링 전 관절 21개를 모두 저장합니다.

공개 데이터 모드에서는 `class_name="hand"`, `handedness=null`입니다. 좌우 라벨을 넣은 사용자 모드에서만 handedness가 출력됩니다.

In [ ]:
SAMPLE_IMAGE = str(image_paths["val"][0])
BOX_CONF = 0.25
KEYPOINT_CONF = 0.5
result = best_model.predict(SAMPLE_IMAGE, imgsz=IMGSZ, conf=BOX_CONF, device=0, verbose=False)[0]
hands = []
canvas = result.orig_img[:, :, ::-1].copy()
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(canvas)
if result.keypoints is not None and len(result.boxes):
    if result.keypoints.conf is None:
        raise RuntimeError("관절 confidence가 없습니다. kpt_shape=[21,3] 학습 결과인지 확인하세요.")
    for i, box in enumerate(result.boxes):
        cls_id = int(box.cls.item())
        points = result.keypoints.xy[i].cpu().numpy()
        confidence = result.keypoints.conf[i].cpu().numpy()
        class_name = result.names[cls_id]
        handedness = class_name if class_name in ("left_hand", "right_hand") else None
        hands.append({
            "class_name": class_name,
            "handedness": handedness,
            "detection_confidence": float(box.conf.item()),
            "bbox_xyxy": box.xyxy[0].cpu().tolist(),
            "landmarks_xy_normalized": result.keypoints.xyn[i].cpu().tolist(),
            "landmark_confidence": confidence.tolist(),
        })
        for a, b in EDGES:
            if min(confidence[a], confidence[b]) >= KEYPOINT_CONF:
                ax.plot(points[[a,b], 0], points[[a,b], 1], color="lime")
        for j, ((x, y), score) in enumerate(zip(points, confidence)):
            if score >= KEYPOINT_CONF:
                ax.scatter(x, y, color="red", s=12)
                ax.text(x+3, y+3, f"{j}:{score:.2f}", color="yellow", fontsize=8)
        x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
        ax.text(x1, y1, f"{class_name} det={float(box.conf.item()):.2f}",
                color="white", bbox=dict(facecolor="black", alpha=0.7))
ax.axis("off")
fig.tight_layout()
fig.savefig(RUN_DIR / "prediction.png", dpi=150)
plt.show()
output = {"image": SAMPLE_IMAGE, "keypoint_names": KEYPOINT_NAMES, "hands": hands}
(RUN_DIR / "prediction.json").write_text(
    json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(output, ensure_ascii=False, indent=2))

## 9. 학습 결과 다운로드
best.pt, last.pt, 학습 설정, 검증 결과, 추론 예시를 ZIP으로 받습니다.
데이터 이미지는 포함하지 않습니다. YAML의 절대 경로는 다른 환경에서 수정해야 합니다.
가중치를 이 프로젝트에서 사용하려면 별도의 YOLO 추론 백엔드 구현이 필요합니다.

In [ ]:
archive_path = shutil.make_archive(str(Path("/content") / (RUN_DIR.name + "_results")),
                                   "zip", root_dir=str(RUN_DIR))
print("결과 폴더:", RUN_DIR)
files.download(archive_path)

## 선택: 중단된 학습 이어서 실행
아래 셀은 기본적으로 실행하지 않습니다. 다시 연결한 경우 1번 설치 셀을 실행하고 Drive를 마운트한 뒤,
데이터 ZIP을 다시 풀어야 합니다. 체크포인트가 참조하는 원래 데이터 경로(학습 args.yaml의 data)를 복구하세요.
`resume=True`는 완료된 학습의 추가 파인튜닝이 아니라 중단된 학습의 복구용입니다.

In [ ]:
RESUME = False
LAST_CHECKPOINT = "/content/drive/MyDrive/hand_pose_runs/실행폴더/weights/last.pt"
if RESUME:
    checkpoint = Path(LAST_CHECKPOINT)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    resumed_model = YOLO(str(checkpoint))
    resumed_model.train(resume=True)